# ThoresT

`Pacman.py` ist die Engine unveraendert. `thorest.py` enthaelt meine Klasse.
Der Import unten traegt sie ein, damit `Field()` sie benutzt.

In [ ]:
import Pacman
import thorest          # registriert ThoresT

print("Field() benutzt:", Pacman.ThoresT.__name__)

## 1. Der Original-Test (100 Zuege, jedes Brett ausgegeben)

In [ ]:
from Pacman import Koordinaten, Direction, Position, Field

field = Field(15)
# print(field)
for i in range(100):
    print(" ")
    for pacman in field.pacmans:
        if pacman.alive:
            pacman.TurnOrMoveOrStill()
    print(field)

## 2. Dieselbe Partie, nur das Ergebnis

In [ ]:
import random
from Pacman import Field

random.seed(1)
feld = Field(15)
ich = feld.pacmans[-1]

for zug in range(100):
    for spieler in feld.pacmans:
        if spieler.alive:
            spieler.TurnOrMoveOrStill()

print(feld)
print()
for spieler in sorted(feld.pacmans, key=lambda p: -p.strength):
    markierung = "   <-- wir" if spieler is ich else ""
    tot = "  (tot)" if not spieler.alive else ""
    print(f"{spieler.name:<10s} {spieler.strength:6.0f}{tot}{markierung}")

gegner = [p.strength for p in feld.pacmans if p is not ich]
print()
print("ThoresT %.0f gegen besten Gegner %.0f  ->  %s"
      % (ich.strength, max(gegner), "SIEG" if ich.strength > max(gegner) else "verloren"))
print("%.2f ms/Zug, Fehler: %d"
      % (ich.brain.total_ms / max(1, ich.brain.turn), ich.brain.faults))

## 3. Mehrere Partien

Eine einzelne Partie sagt bei diesem Spiel wenig — die Kaempfe sind
Wuerfelwuerfe. Erst ueber viele Partien sieht man, wie stark ein Bot ist.

In [ ]:
import random
from Pacman import Field

siege = 0
staerken = []
partien = 20

for seed in range(partien):
    random.seed(seed)
    feld = Field(15)
    ich = feld.pacmans[-1]
    for zug in range(100):
        for spieler in feld.pacmans:
            if spieler.alive:
                spieler.TurnOrMoveOrStill()
    gegner = max(p.strength for p in feld.pacmans if p is not ich)
    staerken.append(ich.strength)
    siege += ich.strength > gegner

print("%d von %d Partien gewonnen (%.0f%%)" % (siege, partien, 100*siege/partien))
print("mittlere Staerke: %.1f" % (sum(staerken)/len(staerken)))
print()
print("Bei sechs gleich starken Spielern waeren ~17% fair.")

## 4. Die Regel, die alles entscheidet

Steht in `Pacman._Move`: **aus welcher Richtung** man angreift, entscheidet
den Kampf — nicht wen.

In [ ]:
from Pacman import Direction

def gewinnchance(a, b, meine_richtung, seine_richtung):
    z = meine_richtung + seine_richtung
    if z._x == 0 and z._y == 0:
        faktor = 1.0        # frontal
    elif abs(z._x) == 1 and abs(z._y) == 1:
        faktor = 0.2        # quer
    else:
        faktor = 0.1        # von hinten
    return a / (a + b * faktor)

print("Bei gleicher Staerke gewinnt der Angreifer mit:")
print("  frontal      %.1f%%" % (100*gewinnchance(10, 10, Direction.east, Direction.west)))
print("  quer         %.1f%%" % (100*gewinnchance(10, 10, Direction.east, Direction.north)))
print("  von hinten   %.1f%%" % (100*gewinnchance(10, 10, Direction.east, Direction.east)))
print()
print("Umgekehrt: wenn mich jemand angreift, bestimmt MEINE Blickrichtung")
print("seine Chance. Ihm entgegenzuschauen drueckt sie von 91% auf 50%.")